# Geopack Python SDK: Async Client (`AsyncGeopackClient`)

This notebook demonstrates **non-blocking** access to the Geopack API using `httpx` and `asyncio`.

**When to use async**
- Poll several background tasks in parallel (`wait_for_tasks`)
- Fetch datasets, datastores, quotas, and task summary concurrently
- Integrate the SDK inside an `asyncio` service or Jupyter with top-level `await`

**Still use sync `GeopackClient` for**
- Multipart dataset upload
- Streaming download of workflow artifacts / generated files
- GeoPandas `read_dataset`

---

### Prerequisites
1. Running Geoportal API (e.g. `http://localhost:3000/api`)
2. `.env` in this `notebooks/` folder (or repo root) with `GEOPACK_API_URL`, `GEOPACK_USERNAME`, `GEOPACK_PASSWORD`
3. `httpx` installed: `pip install httpx` or `pip install geopack-sdk[async]`

In [ ]:
# Install async dependency (safe to re-run)
%pip install httpx python-dotenv -q

## 1. Bootstrap SDK + `AsyncGeopackClient`

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import os
import sys
import time

from dotenv import load_dotenv

try:
    current_dir = os.getcwd()
    source_path = os.path.abspath(os.path.join(current_dir, "..", "src"))
    if os.path.exists(source_path) and source_path not in sys.path:
        sys.path.insert(0, source_path)
        print(f"Using SDK from local source: {source_path}")
    else:
        print("Using SDK from site-packages")
except Exception:
    print("Could not adjust sys.path; using installed package")

from geopack_sdk import AsyncGeopackClient
from geopack_sdk.tasks import task_message_badge_severity

load_dotenv()
API_URL = os.getenv("GEOPACK_API_URL", "http://localhost:3000/api")
print(f"API URL: {API_URL}")

## 2. Login (async)

In [ ]:
client = AsyncGeopackClient(base_url=API_URL)

await client.auth.login()
me = await client.users.me()
print(f"Logged in as {me.userName} (id={me.id})")

## 3. Parallel API calls with `asyncio.gather`

Four independent endpoints are fetched **at the same time** instead of one after another.

In [ ]:
t0 = time.perf_counter()
summary, datasets_page, stores, quota = await asyncio.gather(
    client.tasks.summary(),
    client.datasets.list(page_size=5),
    client.datastores.list(),
    client.quotas.my_summary(),
)
elapsed = time.perf_counter() - t0

print(f"Active tasks: pending={summary.pending}, processing={summary.processing}")
print(f"Datasets (first page): {len(datasets_page.datasets)}")
for ds in datasets_page.datasets[:3]:
    print(f"  - {ds.name} [{ds.dataType}] id={ds.id}")
print(f"Datastores: {len(stores.datastores)}")
print(f"Quotas enabled: {quota.enabled}")
print(f"Elapsed (gather): {elapsed:.2f}s")

## 4. Poll a single background task

Set `TEST_TASK_ID` in `.env` to a real task UUID from the portal (upload, export, workflow, etc.).

In [ ]:
TASK_ID = os.getenv("TEST_TASK_ID", "").strip()

if not TASK_ID:
    print("Set TEST_TASK_ID in .env to run this cell.")
else:
    task = await client.tasks.wait_for_task(TASK_ID, timeout=300, interval=2)
    severity = task_message_badge_severity(task)
    print(f"Task {TASK_ID}: status={task.status}, log badge={severity}")

## 5. Poll multiple tasks in parallel

Set `TEST_TASK_IDS=id1,id2` in `.env` (comma-separated).

In [ ]:
TASK_IDS = [x.strip() for x in os.getenv("TEST_TASK_IDS", "").split(",") if x.strip()]

if len(TASK_IDS) < 2:
    print("Set TEST_TASK_IDS=id1,id2 in .env to run parallel polling.")
else:
    results = await client.tasks.wait_for_tasks(TASK_IDS, timeout=300, interval=2, quiet=True)
    for t in results:
        print(f"{t.taskId}: {t.status} (badge={task_message_badge_severity(t)})")

## 6. Context manager / cleanup

Always close the underlying `httpx` client when you are done (or use `async with`).

In [ ]:
await client.aclose()
print("Client closed.")

# Alternative for scripts:
# async with AsyncGeopackClient(base_url=API_URL) as client:
#     await client.auth.login()
#     ...

## 7. Script equivalent

From the repo root:

```bash
cd python-sdk
pip install httpx python-dotenv
set PYTHONPATH=src
python test_async_sdk.py
```

Optional env vars: `TEST_TASK_ID`, `TEST_TASK_IDS`.